##### **Analsing Data With Pandas**

In [1]:
import pandas as pd

##### **Load The Clean Data**

In [2]:
df= pd.read_csv("Datasets/kakuma_fresh_mart_sales_clean.csv", parse_dates=['date'])

In [3]:
df.head()

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price,is_return,unsual_qty,new_customer
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,False,True
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,False,True
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,False,True
3,R10004,2026-10-01,Kitale,Milk,dairy,2.0,65.0,Cash,C0308,False,False,False,False
4,R10005,NaT,Kakuma Town,Detergent,household,3.0,270.0,Cash,NaN,False,False,False,True


In [4]:
# raw and columns
df.shape

(2467, 13)

In [5]:
# revenue a create column
df['revenue']= df['qty'] * df['price']

df.head(3)

,receipt,date,branch,product,category,qty,price,payment,customer,imputed_price,is_return,unsual_qty,new_customer,revenue
0,R10001,2026-10-01,Kakuma Town,Water,drinks,2.0,50.0,M-Pesa,NaN,False,False,False,True,100.0
1,R10002,2026-10-01,Lodwar,Water,drinks,4.0,50.0,M-Pesa,NaN,False,False,False,True,200.0
2,R10003,2026-10-01,Kakuma Town,Salt,cooking,2.0,40.0,Cash,NaN,False,False,False,True,80.0


In [6]:
# remove unsual quantity(999, 500, 250)
sales= df[df['unsual_qty']==False]

# Net revenue
print("Net Revenue:",sales['revenue'].sum())

Net Revenue: 949214.0


##### **Where Does the Revenue Come From**

** Question 1 ** which `branch`  brings inthe most revenue, and is it because
customers `spend more` or because there are `are more of them`

In [7]:
# Revenue by branch
sales.groupby('branch')['revenue'].sum().sort_values(ascending=False)

branch
Kakuma Town    375795.0
Lodwar         287493.0
Kitale         285926.0
Name: revenue, dtype: float64

`Kakuma Town` has the highest revenue `(375795.0)`- `40%` compare to Lodwar 
(287493.0) and kitale (285926.0)

In [8]:
# Branch summary- Revenue, order, AOV
bransch_summary= sales.groupby('branch').agg(
    total_revenue=('revenue', 'sum'),
    customers=('receipt', 'count'),
    AVO= ('revenue', 'mean')
)
bransch_summary.sort_values('total_revenue', ascending= False).reset_index()

,branch,total_revenue,customers,AVO
0,Kakuma Town,375795.0,1000,375.795000
1,Lodwar,287493.0,727,395.451169
2,Kitale,285926.0,737,387.959294


**Kakuma Town earns about 40% of revenue (KES 375,795 of 949,214), but because it
sells more, not because customers spend more.** It has 1,000 sales lines against about
730 elsewhere, and its average per line (KES 376) is the lowest of the three branches.

##### **3. What Drives Revenue**

 question 2. which categories and product drive. Focuson a share over amount

In [9]:
# Revenue by Category
by_category = sales.groupby("category")["revenue"].sum().sort_values(ascending=False)
share = (by_category / by_category.sum() * 100).round(1)

##### **Revenue vs Volume**

In [10]:
# order by product
product_orders= sales.groupby('product')['qty'].sum().sort_values(ascending= False)

# Top 10
product_orders.head(10)

product
Water          552.0
Soda           505.0
Milk           500.0
Wheat flour    425.0
Sugar          413.0
Cooking oil    401.0
Maize flour    362.0
Rice           328.0
Soap           318.0
Eggs           298.0
Name: qty, dtype: float64

** VRevenue is concentrated: five of 19 products bring in 54% of revenue, and Grains,
Cooking and Dairy together make up 72%.** Eggs is the top earner (about 15% of
revenue) although it ranks only tenth by units sold, while Water is the top seller by
units.

##### **4: Do Customers Behave Differently?**

**Question 3:** Do loyalty members spend differently from other shoppers, and how do
customers pay?

In [11]:
# New vs Returning customers
sales['new_customer'].value_counts()

new_customer
True     1367
False    1097
Name: count, dtype: int64

In [12]:
# customers Summary
customers_summary= sales.groupby('new_customer')['revenue'].agg(['count', 'mean', 'median'])

customers_summary.round(2).reset_index()

,new_customer,count,mean,median
0,False,1097,386.39,250.0
1,True,1367,384.31,260.0


Returning average KES 386 per line and non-members KES 384. The medians are 250 and
260. These are practically the same. There is no sign that loyalty members spend more
per line.

##### **Payment Behaviour**

In [13]:
# Payment method in different branches
branch_payment= pd.crosstab(sales['branch'], sales['payment'], normalize= 'index')*100

branch_payment.round(1)

payment,Card,Cash,Credit,M-Pesa,unknown
branch,,,,,
Kakuma Town,5.9,45.6,3.6,43.0,1.9
Kitale,7.7,46.3,2.6,40.3,3.1
Lodwar,7.3,41.7,3.6,44.8,2.6


Loyalty members do not spend more per sales line: KES 386 against 384 on average,
and medians of 250 against 260, with over 1,000 lines in each group. Payment habits
are similar across branches: cash and M-Pesa each account for about 40 to 46% of
lines. So what: the loyalty scheme does not currently lift spending per item, so it is

worth asking what it is meant to achieve. M-Pesa is nearly as important as cash, so M-
Pesa records need to be reliable. Caveat: the file records lines, not visits, so members

may still visit more often. 

##### **5: When do we sell most**
Question 4 when do we  sell most

##### **a Daily Revenue**


In [14]:
# daily revenue
daily= sales.groupby('date')['revenue'].sum().reset_index()

# Get days names
daily['weekday']= daily['date'].dt.day_name()

daily.head(10)


,date,revenue,weekday
0,2026-10-01,32474.0,Thursday
1,2026-10-02,40125.0,Friday
2,2026-10-03,46930.0,Saturday
3,2026-10-04,47908.0,Sunday
4,2026-10-05,40676.0,Monday
5,2026-10-06,45442.0,Tuesday
6,2026-10-07,41730.0,Wednesday
7,2026-10-08,34588.0,Thursday
8,2026-10-09,36341.0,Friday
9,2026-10-10,56629.0,Saturday


In [15]:
# Daily sales(7 days)
days_order=['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_sales= daily.groupby('weekday')['revenue'].mean().reindex(days_order)

daily_sales.reset_index().round(2)


,weekday,revenue
0,Monday,39790.00
1,Tuesday,40687.33
2,Wednesday,38196.67
3,Thursday,32750.67
4,Friday,37236.67
5,Saturday,51206.33
6,Sunday,53062.33


In [16]:
# compare weekend vs week days avg revenue
daily['is_weekend']=daily['weekday'].isin(['Saturday', 'Sunday'])

daily.groupby('is_weekend')['revenue'].sum().reset_index().round(2)

,is_weekend,revenue
0,False,565984.0
1,True,312806.0


##### **Do weekends perform better in ALL branches**

In [17]:
sales['is_weekend']= sales['date'].dt.day_of_week >=5


pvt= pd.pivot_table(
    sales,
    index= 'branch',
    columns= 'is_weekend',
    values= 'revenue',
    aggfunc= 'sum'
)
pvt.round(2)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13560\426113305.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['is_weekend']= sales['date'].dt.day_of_week >=5


is_weekend,False,True
branch,,
Kakuma Town,254019.0,121776.0
Kitale,189197.0,96729.0
Lodwar,193192.0,94301.0


In [18]:
days_per_group= sales.groupby('is_weekend')['date'].nunique()

per_day=pvt/days_per_group

per_day.round(0)

is_weekend,False,True
branch,,
Kakuma Town,16935.0,20296.0
Kitale,12613.0,16122.0
Lodwar,12879.0,15717.0


> **Weekend days earn about 41% more than weekdays: KES 57,141 a day against
40,425. The lift is present in every branch (38% at Kakuma Town, 42% at Kitale, 45% at
Lodwar).**

> **So what:** schedule more staff and make sure stock is full for Saturday and
Sunday. 

> **Caveat:** the data covers only three weeks, with each weekday seen three
times, so differences between individual weekdays are weak evidence.

## Findings summary
**Bottom line:** Sales are not weak at any one branch. The business is concentrated
in a few products and in the weekend, so staffing and stock should follow those.
1. Kakuma Town earns about 40% of revenue because of volume, not higher spending.
2. Five of 19 products bring in 54% of revenue. Eggs leads by revenue, not by units.
3. Loyalty members do not spend more per line, and payments are alike everywhere.
4. Weekend days earn about 41% more than weekdays, in every branch.
**Please confirm:** the three extreme quantities, and day-first slash dates.
**Not covered:** customers and visits, because the file records items only.